In [ ]:
from pathlib import Path

import torch
from PIL import Image

from infer import load_model, preprocess, transcribe, transcribe_beam

In [ ]:
CHECKPOINT = "runs/latest_run/best_model.pt"  # train.py keeps runs/latest_run pointing at the newest run

DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

# without the KenLM at KENLM_PATH in infer.py, beam search runs without an LM
model, tokenizer = load_model(CHECKPOINT, DEVICE)
print(f"Loaded on {DEVICE} | backbone={model.backbone_name} | vocab={len(tokenizer)} | "
      f"decoder={model.use_decoder} | LM={tokenizer.kenlm_path is not None} @ w={tokenizer.lm_weight}")

## Transcribe one line image

`transcribe_beam` runs CTC beam search with a single KenLM over all training domains fused in, weighted by `LM_WEIGHT`. `greedy` is the no-LM reference. The autoregressive decoder is left out because it adds almost nothing once the LM is in the beam. For mostly numeric or tabular text, lower `LM_WEIGHT` toward 0.5.

In [ ]:
IMAGE_PATH = Path('data/valid/example_line.png')

img = Image.open(IMAGE_PATH).convert("RGB")
print(f"{IMAGE_PATH.name} | {img.size[0]}x{img.size[1]}")
display(img)

beam = transcribe_beam(model, tokenizer, [img], DEVICE)[0]
print("beam+LM :", beam)

greedy = transcribe(model, tokenizer, [img], DEVICE, use_decoder=False)[0]
print("greedy  :", greedy)

## Per-patch maps

The heatmaps from TensorBoard's `viz/` tab (confidence, entropy, feature norms, expert routing), overlaid on the model input. `span_mask` is skipped: it is only applied during training, so at inference it is a random mask with no effect.

In [ ]:
tensor = preprocess(img, model).unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    _, _, viz_out, _ = model.forward(tensor, return_attention=True)

print(f"viz outputs: {len(viz_out)}")
for k, v in viz_out.items():
    if torch.is_tensor(v) and v.numel() == 1:
        print(f"  {k}: {v.item():.4f}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

MEAN = torch.tensor((0.485, 0.456, 0.406)).view(-1, 1, 1)
STD = torch.tensor((0.229, 0.224, 0.225)).view(-1, 1, 1)
CMAPS = {"confidence_map": "viridis", "entropy_map": "plasma"}
SKIP_MAPS = {"span_mask"}


def denormalize(t):
    x = t.detach().float().cpu() * STD + MEAN
    return ((x - x.min()) / (x.max() - x.min() + 1e-8)).clamp(0, 1)


disp = denormalize(tensor[0])
_, H, W = disp.shape
maps = [(k, v) for k, v in viz_out.items()
        if torch.is_tensor(v) and v.ndim >= 3 and k not in SKIP_MAPS]

fig, axes = plt.subplots(len(maps) + 1, 1, figsize=(16, 1.9 * (len(maps) + 1)))
axes[0].imshow(disp.permute(1, 2, 0).numpy())
axes[0].set_title("input", fontsize=10, loc="left")
axes[0].axis("off")

for ax, (name, hm_tensor) in zip(axes[1:], maps):
    hm = (hm_tensor[0, 0] if hm_tensor.ndim == 4 else hm_tensor[0]).detach().float().cpu().nan_to_num()
    if "expert_routing" in name:
        hm = hm / max(1, model.num_experts - 1)
        cname = "tab20" if model.num_experts > 10 else "tab10"
    else:
        hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
        cname = CMAPS.get(name, "magma")
    hm_up = torch.nn.functional.interpolate(hm[None, None], size=(H, W), mode="nearest")[0, 0]
    rgb = torch.tensor(plt.get_cmap(cname)(hm_up.numpy())[:, :, :3], dtype=torch.float32).permute(2, 0, 1)
    overlay = 0.5 * disp + 0.5 * rgb
    ax.imshow(overlay.permute(1, 2, 0).numpy())
    ax.set_title(name, fontsize=10, loc="left")
    ax.axis("off")

plt.tight_layout()